In [1]:
import pandas as pd

# Reemplaza 'ruta/al/archivo.xlsx' con la ruta real de tu archivo en Google Drive
file_path = '/content/drive/MyDrive/Colab Notebooks/Emitido_24092025-3.xlsx'

# Leer el archivo Excel
df = pd.read_excel(file_path)

# Mostrar las primeras filas del dataframe para verificar que se leyó correctamente
display(df.head())


,NRO LOTE,NRO LOTES ANTERIORES,FECHA CARGA LOTE,CÓDIGO PRODUCTO,PRODUCTO,PLAN,NOMBRE DE PLAN,CÓD. CERTIFICADO CANAL,TIPO MOVIMIENTO,NOMBRE,...,NUMERO DE POLIZA AE,FEC. INICIO AE,FEC. FIN AE,PRIMA BRUTA AE,MONTO COMISIÓN CANAL,FEC. INICIO AX,FEC. FIN AX,PRIMA BRUTA AX,TIPO DOC EMISIÓN ORIGEN,N. DOC EMISIÓN ORIGEN
0,1338999,NaN,09/09/2025 10:56:55,4492,Continental Vida Renta c/Devolución,398850,Vida Renta plan único 2022,C00110368804000456451,Poliza Nueva,FIORELLA,...,124438739,01/09/2025,01/09/2030,185.5,0,NaN,NaN,NaN,NaN,NaN
1,1339001,NaN,09/09/2025 10:56:57,4492,Continental Vida Renta c/Devolución,184329,VIDA RENTA BBVA DEV (USD),C00117799444000017718,Renovacion,ROXANA VIVIANA,...,81388455,06/03/2021,06/03/2031,12.5,0,NaN,NaN,NaN,NaN,NaN
2,1339001,NaN,09/09/2025 10:56:57,4492,Continental Vida Renta c/Devolución,184329,VIDA RENTA BBVA DEV (USD),C00110093284000010084,Renovacion,MERI,...,20101993,30/03/2016,30/03/2026,24.0,0,NaN,NaN,NaN,NaN,NaN
3,1339001,NaN,09/09/2025 10:56:57,4492,Continental Vida Renta c/Devolución,184329,VIDA RENTA BBVA DEV (USD),C00110117954000600700,Renovacion,EPIFANIA NINFA,...,107154506,31/07/2020,31/07/2030,97.5,0,NaN,NaN,NaN,NaN,NaN
4,1339001,NaN,09/09/2025 10:56:57,4492,Continental Vida Renta c/Devolución,184329,VIDA RENTA BBVA DEV (USD),C00110126014000213080,Renovacion,RUBEN AMADEO,...,31634081,01/08/2017,01/08/2027,8.5,0,NaN,NaN,NaN,NaN,NaN


In [2]:
# Asegurarse de que los nombres de columna estén en minúsculas para evitar problemas de case-sensitivity
df.columns = df.columns.str.lower()

# df.loc[df['código producto'].isin([6377, 5520]), 'prima bruta'] *= 1.2154
# Agrupar y sumar 'PRIMA BRUTA AE' por 'NOMBRE ARCHIVO', 'NRO LOTE', y 'MONEDA'
suma_prima = df.groupby(['nombre archivo', 'nro lote', 'moneda'])['prima bruta'].sum().unstack(fill_value=0).reset_index()

# Agrupar y contar registros por 'NOMBRE ARCHIVO', 'NRO LOTE', y 'MONEDA'
conteo_registros = df.groupby(['nombre archivo', 'nro lote', 'moneda']).size().unstack(fill_value=0).reset_index()

# Renombrar las columnas adecuadamente
suma_prima.columns.name = None
conteo_registros.columns.name = None
suma_prima = suma_prima.rename(columns={'SOL': 'PRIMA BRUTA SOL', 'USD': 'PRIMA BRUTA USD'})
conteo_registros = conteo_registros.rename(columns={'SOL': 'Registros SOL', 'USD': 'Registros USD'})

# Combinar las dos tablas en una
final_result = pd.merge(suma_prima, conteo_registros, on=['nombre archivo', 'nro lote'])

# Asegurarse de que todos los valores NaN sean reemplazados por 0
final_result = final_result.fillna(0)

# Convertir todos los registros de 'nombre archivo' a minúsculas
final_result['nombre archivo'] = final_result['nombre archivo'].str.lower()

# Mostrar el resultado
display(final_result)

,nombre archivo,nro lote,PRIMA BRUTA SOL,PRIMA BRUTA USD,Registros SOL,Registros USD
0,20100130204_0206001_20240215_005.txt,1341081,0.00,93.00,0,1
1,20100130204_0206001_20250829_004.txt,1341048,0.00,1606.35,0,104
2,20100130204_0206001_20250901_004.txt,1341052,0.00,2348.64,0,185
3,20100130204_0206001_20250902_004.txt,1341058,0.00,660.15,0,43
4,20100130204_0206001_20250903_004.txt,1341065,0.00,676.14,0,44
...,...,...,...,...,...,...
188,20100130204_4121001_20250918_004.txt,1341830,53533.99,0.00,97,0
189,20100130204_4121001_20250919_001.txt,1342940,21385.96,0.00,27,0
190,20100130204_4121001_20250919_004.txt,1342795,59097.59,0.00,102,0
191,20100130204_4121001_20250922_001.txt,1342941,27027.23,0.00,39,0


In [9]:
#!pip install jira
from datetime import datetime
from jira import JIRA, JIRAError
import pandas as pd
import requests
from requests.auth import HTTPBasicAuth
import json

In [10]:
# Autenticación en Jira
usuario = "paul.sanchez@rimac.com.pe"
token = "ATATT3xFfGF0CjokoacwWkFHhMkRb3hOK1mPm_1Obg4sg9GnJSjOeJgxEv54sjvmlcOFlIy9KQ5yvmLFO1b3IK8e9cGEj46QFzb3xwsO47mFyZGMm_jIiexDYOmTava_LgSKUtHUy5hKxkweG4YmHqx91mw3GehtIrbYfnpPV2RLxQFCtozFFqY=342FD831"
server = "https://rimacseguros.atlassian.net/"
jira = JIRA(server, basic_auth=(usuario, token))
#proyecto_jira = 'PMR'
proyecto_jira = 'HSP'

# Autenticación básica
auth = HTTPBasicAuth(usuario, token)
# URL de la API de búsqueda
#url_busqueda = f"{server}/rest/api/3/search"
url_busqueda = f"{server}/rest/api/3/search/jql"
# Headers
headers = {
    "Accept": "application/json"
}

In [11]:
#####################################
######OBTENER ISSUES JQL#############
#####################################
jql = f'project = {proyecto_jira} AND "fecha recepción[date]" >= "2024-10-01" AND "fecha recepción[date]" <= "2025-09-30"'
# jql = f'project = PMR AND "fecha recepción[date]" >= "2024-09-18" AND "fecha recepción[date]" <= "2024-09-30" AND assignee = 712020:dd12a73a-8046-4383-a6c0-20f3a7c64c9e'

params = {
    "jql": jql,
    "fields": "key,summary",  # Necesitamos los keys y los summaries de los issues
    "maxResults": 100,  # Máximo por página, puedes ajustarlo según tus necesidades
    "startAt": 0
}
all_issues = []

while True:
    response = requests.get(url_busqueda, headers=headers, params=params, auth=auth)
    if response.status_code != 200:
        print(f"Error al obtener los issues: {response.status_code}")
        print(response.text)
        break
    response_data = response.json()
    all_issues.extend(response_data['issues'])
    if len(response_data['issues']) < params['maxResults']:
        break
    params['startAt'] += params['maxResults']

print(f"Total issues obtenidos: {len(all_issues)}")

# Mostrar los primeros issues
issue_data = [{"key": issue["key"], "summary": issue["fields"]["summary"]} for issue in all_issues]
print("Issues en el proyecto:")
for issue in issue_data:
    print(f"Key: {issue['key']}, Summary: {issue['summary']}")


Total issues obtenidos: 0
Issues en el proyecto:


In [12]:
# ============================
# OBTENER ISSUES POR JQL (Natalia) - Colab
# ============================

import re
import pandas as pd
from jira import JIRA
from jira.exceptions import JIRAError

# ========= CONFIG =========
JIRA_URL   = "https://rimacseguros.atlassian.net"
USERNAME   = "naguilarz@rimac.com.pe"             # Usuario de Natalia
API_TOKEN  = "ATATT3xFfGF0CjokoacwWkFHhMkRb3hOK1mPm_1Obg4sg9GnJSjOeJgxEv54sjvmlcOFlIy9KQ5yvmLFO1b3IK8e9cGEj46QFzb3xwsO47mFyZGMm_jIiexDYOmTava_LgSKUtHUy5hKxkweG4YmHqx91mw3GehtIrbYfnpPV2RLxQFCtozFFqY=342FD831"    # ⚠️ Pega aquí el token (luego pásalo a entorno seguro)
PROYECTO_JIRA = "PMR"

# Rango de fechas (formato yyyy-MM-dd). Ajusta según necesites.
FECHA_INICIO = "2025-01-01"
FECHA_FIN    = "2025-09-30"

# Nombre visible del campo fecha en JQL.
# En la UI a veces aparece con sufijo [date]; esto puede o no aceptar el API.
# Te doy 3 variantes abajo: con [date], sin [date] y por customfield ID si lo conoces.
CAMPO_FECHA_CON_SUF = 'fecha recepción[date]'  # tal como lo pegaste
CAMPO_FECHA_SIN_SUF = 'fecha recepción'        # sin el sufijo (muchas veces así sí funciona)
CUSTOMFIELD_ID      = None                      # e.g. "customfield_12345" si lo conoces; si no, deja None

# ========= CONEXIÓN =========
jira = JIRA(JIRA_URL, basic_auth=(USERNAME, API_TOKEN))

def construir_jql_por_campo(nombre_campo_o_id):
    """
    Devuelve un JQL que filtra por el campo de fecha indicado (entre FECHA_INICIO y FECHA_FIN).
    Si el campo tiene espacios, lo ponemos entre comillas.
    """
    if nombre_campo_o_id.startswith("customfield_"):
        # Los customfield no van entre comillas
        campo = nombre_campo_o_id
    else:
        # Nombre visible: comillas si contiene espacios o caracteres especiales
        campo = f'"{nombre_campo_o_id}"'
    jql = (
        f'project = {PROYECTO_JIRA} '
        f'AND {campo} >= "{FECHA_INICIO}" AND {campo} <= "{FECHA_FIN}"'
    )
    return jql

def buscar_issues_con_fallbacks():
    """
    Intenta buscar usando (en orden):
    1) CAMPO_FECHA_CON_SUF (p.ej. "fecha recepción[date]")
    2) CAMPO_FECHA_SIN_SUF (p.ej. "fecha recepción")
    3) CUSTOMFIELD_ID (p.ej. customfield_12345) si está configurado

    Retorna (issues, jql_usado)
    """
    errores = []

    # Intento 1: nombre con sufijo [date]
    if CAMPO_FECHA_CON_SUF:
        jql = construir_jql_por_campo(CAMPO_FECHA_CON_SUF)
        try:
            issues = jira.search_issues(jql_str=jql, maxResults=False, fields="summary")
            if issues is not None:
                return issues, jql
        except JIRAError as e:
            errores.append(("CON_SUF", jql, str(e)))

    # Intento 2: nombre sin sufijo
    if CAMPO_FECHA_SIN_SUF:
        jql = construir_jql_por_campo(CAMPO_FECHA_SIN_SUF)
        try:
            issues = jira.search_issues(jql_str=jql, maxResults=False, fields="summary")
            if issues is not None:
                return issues, jql
        except JIRAError as e:
            errores.append(("SIN_SUF", jql, str(e)))

    # Intento 3: por customfield ID si lo conoces
    if CUSTOMFIELD_ID:
        jql = construir_jql_por_campo(CUSTOMFIELD_ID)
        try:
            issues = jira.search_issues(jql_str=jql, maxResults=False, fields="summary")
            if issues is not None:
                return issues, jql
        except JIRAError as e:
            errores.append(("CUSTOMFIELD_ID", jql, str(e)))

    # Si llegamos aquí, fallaron todos los intentos
    msg = ["No se pudo ejecutar la búsqueda JQL con ninguna variante. Detalle de errores:"]
    for origen, j, err in errores:
        msg.append(f"- Variante {origen} | JQL: {j}\n  Error: {err[:500]}")
    raise RuntimeError("\n".join(msg))

# ========= BÚSQUEDA =========
issues, jql_usado = buscar_issues_con_fallbacks()
print("JQL usado:", jql_usado)
print(f"Total issues obtenidos: {len(issues)}")

# ========= ARMAR SALIDA (key, summary) =========
issue_data = [{"key": it.key, "summary": getattr(it.fields, "summary", None)} for it in issues]

print("\nIssues (muestra de 10):")
for it in issue_data[:10]:
    print(f"Key: {it['key']}, Summary: {it['summary']}")

# (Opcional) DataFrame si lo quieres manipular
df_issues = pd.DataFrame(issue_data)
df_issues.head()

JQL usado: project = PMR AND "fecha recepción[date]" >= "2025-01-01" AND "fecha recepción[date]" <= "2025-09-30"
Total issues obtenidos: 0

Issues (muestra de 10):


""


In [13]:
#####################################
######ACTUALIZAR  ISSUES#############
#####################################

# Crear un diccionario para buscar issues por summary
issue_dict = {issue["fields"]["summary"]: issue["key"] for issue in all_issues}
num_issues = 0
# Actualizar issues en Jira
for _, row in final_result.iterrows():
    nombre_archivo = row['nombre archivo']
    if nombre_archivo in issue_dict:
        print(f"Actualizando issue {issue_dict[nombre_archivo]}")
        num_issues += 1
        issue_key = issue_dict[nombre_archivo]
        update_url = f"{server}/rest/api/3/issue/{issue_key}"

        # Crear el payload de actualización
        update_payload = {
            "fields": {
                "customfield_10752": row["Registros SOL"],#'Total Registros Emitidos PEN'
                "customfield_10755": row["PRIMA BRUTA SOL"],#'Total Prima Bruta Emitida PEN'
                "customfield_10507": row["Registros USD"],#'Total Registros Emitidos USD'
                "customfield_10536": row["PRIMA BRUTA USD"] #'Total Prima Bruta Emitida USD'
            }
        }

        # Realizar la solicitud de actualización
        response = requests.put(update_url, headers=headers, json=update_payload, auth=auth)
        if response.status_code == 204:
            print(f"Issue {issue_key} actualizado correctamente.")
        else:
            print(f"Error al actualizar el issue {issue_key}: {response.status_code}")
            print(response.text)
print(f"Issues actualizados: {num_issues}")
print("Actualización completa.")



NameError: name 'final_result' is not defined

In [ ]:
# ============================
# ACTUALIZAR ISSUES EN JIRA (Natalia) - Colab
# ============================

import re
import pandas as pd
from jira import JIRA

# ========= CONFIG =========
JIRA_URL   = "https://rimacseguros.atlassian.net"
USERNAME   = "naguilarz@rimac.com.pe"             # Usuario de Natalia
API_TOKEN  = "ATATT3xFfGF0CjokoacwWkFHhMkRb3hOK1mPm_1Obg4sg9GnJSjOeJgxEv54sjvmlcOFlIy9KQ5yvmLFO1b3IK8e9cGEj46QFzb3xwsO47mFyZGMm_jIiexDYOmTava_LgSKUtHUy5hKxkweG4YmHqx91mw3GehtIrbYfnpPV2RLxQFCtozFFqY=342FD831"    # ⚠️ Pegar token aquí (luego mover a entorno seguro)
PROJECT_KEY = "PMR"

# Filtros opcionales para limitar la búsqueda JQL:
FILTRAR_POR_PRODUCTO = False     # Cambia a True si quieres filtrar por producto
NOMBRE_CAMPO_PRODUCTO = 'Producto'  # Nombre del campo tal como aparece en JQL (no el ID). Ajusta si difiere.
PRODUCTO_FILTRO = 'Multirriesgo Negocio'  # Valor exacto del producto en Jira (ajústalo)
VENTANA_DIAS    = 45             # Solo issues creados en los últimos N días. Pon 0 para desactivar.

# Ruta del Excel con columnas: nombre archivo, Registros SOL, PRIMA BRUTA SOL, Registros USD, PRIMA BRUTA USD
# Si ya tienes un DataFrame llamado final_result en memoria, el script lo usará. Si no, leerá este Excel.
RUTA_EXCEL = "/content/drive/MyDrive/Tramas_Diarias/Resultados/final_result.xlsx"

# ========= CONEXIÓN =========
jira = JIRA(JIRA_URL, basic_auth=(USERNAME, API_TOKEN))

# ========= CONSTRUIR JQL =========
jql_parts = [f'project = {PROJECT_KEY}']
if FILTRAR_POR_PRODUCTO and PRODUCTO_FILTRO:
    # Importante: el filtro por producto usa el NOMBRE del campo en JQL (no el ID).
    # Asegúrate que el campo se llama "Producto" en tu Jira. Si tiene otro nombre visible, cámbialo en NOMBRE_CAMPO_PRODUCTO.
    jql_parts.append(f'"{NOMBRE_CAMPO_PRODUCTO}" = "{PRODUCTO_FILTRO}"')

if VENTANA_DIAS and VENTANA_DIAS > 0:
    jql_parts.append(f'created >= -{VENTANA_DIAS}d')

# Opcional: limitar a tipo Task si aplica
jql_parts.append('issuetype = Task')

JQL = ' AND '.join(jql_parts)
print("JQL usado:", JQL)

# ========= BUSCAR ISSUES =========
# maxResults=False pagina internamente y trae todo lo que cumpla el JQL
issues = jira.search_issues(jql_str=JQL, maxResults=False, fields="summary")
print(f"Se encontraron {len(issues)} issues que cumplen el JQL.")

# ========= Normalizador de textos (para evitar mismatches por espacios/case) =========
def normalize(s):
    if not isinstance(s, str):
        return s
    s = s.strip()
    s = re.sub(r'\s+', ' ', s)  # colapsa espacios
    return s.casefold()         # caseless

# ========= Mapeo summary -> key =========
summary_to_key = {}
duplicados = {}

for it in issues:
    key = it.key
    summary = getattr(it.fields, "summary", None)
    if not summary:
        continue
    n = normalize(summary)
    if n in summary_to_key:
        # Detecta duplicados por summary normalizado
        duplicados.setdefault(n, []).append(key)
    summary_to_key[n] = key

if duplicados:
    print(f"⚠️ Encontrados {len(duplicados)} summaries duplicados (normalizados). Se usará el último cargado para cada uno.")
    # Si necesitas, imprime algunos para revisar:
    cnt = 0
    for k,v in duplicados.items():
        print(f"  - '{k}': {v}")
        cnt += 1
        if cnt >= 5:
            print("  ...")
            break

# ========= Cargar datos: final_result =========
# Si ya existe un DataFrame llamado final_result, úsalo; si no, lee del Excel
try:
    final_result
    print("Usando DataFrame 'final_result' ya presente en memoria.")
except NameError:
    print("No se encontró 'final_result' en memoria; leyendo desde Excel...")
    final_result = pd.read_excel(RUTA_EXCEL)

# Validación de columnas requeridas
columnas_requeridas = ['nombre archivo', 'Registros SOL', 'PRIMA BRUTA SOL', 'Registros USD', 'PRIMA BRUTA USD']
faltantes = [c for c in columnas_requeridas if c not in final_result.columns]
if faltantes:
    raise ValueError(f"Faltan columnas en el dataset: {faltantes}. Asegúrate que tu Excel/DataFrame tenga exactamente esas columnas.")

# ========= Helper de tipos numéricos =========
def to_number_or_none(v):
    if pd.isna(v):
        return None
    if isinstance(v, str):
        v = v.strip().replace(",", "")
        if v == "":
            return None
    try:
        return float(v)
    except:
        return None

# ========= ACTUALIZAR =========
updated = skipped = not_found = errors = 0

for idx, row in final_result.iterrows():
    nombre_archivo = row.get('nombre archivo')

    if not isinstance(nombre_archivo, str) or not nombre_archivo.strip():
        skipped += 1
        print(f"[Fila {idx}] Sin 'nombre archivo'; se omite.")
        continue

    nname = normalize(nombre_archivo)
    issue_key = summary_to_key.get(nname)

    if not issue_key:
        not_found += 1
        print(f"No se encontró issue con summary '{nombre_archivo}'.")
        continue

    fields_update = {
        "customfield_10752": to_number_or_none(row.get("Registros SOL")),   # Total Registros Emitidos PEN
        "customfield_10755": to_number_or_none(row.get("PRIMA BRUTA SOL")), # Total Prima Bruta Emitida PEN
        "customfield_10507": to_number_or_none(row.get("Registros USD")),   # Total Registros Emitidos USD
        "customfield_10536": to_number_or_none(row.get("PRIMA BRUTA USD"))  # Total Prima Bruta Emitida USD
    }

    # Quita None para no pisar con nulls
    fields_update = {k: v for k, v in fields_update.items() if v is not None}

    if not fields_update:
        skipped += 1
        print(f"{issue_key}: sin datos válidos para actualizar ({nombre_archivo}).")
        continue

    try:
        issue = jira.issue(issue_key)
        issue.update(fields=fields_update)
        updated += 1
        print(f"{issue_key}: actualizado correctamente ({nombre_archivo}).")
    except Exception as e:
        errors += 1
        print(f"{issue_key}: error al actualizar -> {e}")

print("\n=== RESUMEN ===")
print(f"Actualizados: {updated}")
print(f"Omitidos (sin datos): {skipped}")
print(f"No encontrados (summary no coincide): {not_found}")
print(f"Errores: {errors}")



JQL usado: project = PMR AND created >= -45d AND issuetype = Task
Se encontraron 0 issues que cumplen el JQL.
Usando DataFrame 'final_result' ya presente en memoria.
No se encontró issue con summary '20100130204_0206001_20240215_005.txt'.
No se encontró issue con summary '20100130204_0206001_20250829_004.txt'.
No se encontró issue con summary '20100130204_0206001_20250901_004.txt'.
No se encontró issue con summary '20100130204_0206001_20250902_004.txt'.
No se encontró issue con summary '20100130204_0206001_20250903_004.txt'.
No se encontró issue con summary '20100130204_0206001_20250904_004.txt'.
No se encontró issue con summary '20100130204_0206001_20250905_004.txt'.
No se encontró issue con summary '20100130204_0206001_20250908_004.txt'.
No se encontró issue con summary '20100130204_0206001_20250909_004.txt'.
No se encontró issue con summary '20100130204_0238001_20250829_004.txt'.
No se encontró issue con summary '20100130204_0238001_20250901_004.txt'.
No se encontró issue con summar